# Preprocessing

This notebook prepares the TLC trip records and subway station data for
later exploratory analysis and modelling. The raw trip records are first
reduced to zone-hour aggregates so the downstream work remains manageable.

## Project Paths

Define the project folders once so the rest of the notebook can refer to
the same raw, interim, and processed data locations.

In [41]:
from pathlib import Path

PROJECT_ROOT = Path("/mnt/d/ADS_proj1")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

YELLOW_DIR = RAW_DIR / "tlc" / "yellow"
GREEN_DIR = RAW_DIR / "tlc" / "green"
HVFHV_DIR = RAW_DIR / "tlc" / "hvfhv"

MTA_PATH = RAW_DIR / "external" / "mta_subway_stations.csv"
ZONE_LOOKUP_PATH = RAW_DIR / "zones" / "taxi_zone_lookup.csv"
TAXI_ZONES_PATH = RAW_DIR / "zones" / "taxi_zones" / "taxi_zones.shp"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Spark Setup

Start a local Spark session. The extra memory and adaptive execution
settings help with the large HVFHV files.

In [42]:
from functools import reduce

import numpy as np
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("MAST30034 Project 1")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Raw File Check

Record which monthly files are available for each service type. This is a
lightweight check before reading the data.

In [43]:
raw_paths = {
    "yellow": sorted(YELLOW_DIR.glob("yellow_tripdata_2024-*.parquet")),
    "green": sorted(GREEN_DIR.glob("green_tripdata_2024-*.parquet")),
    "hvfhv": sorted(HVFHV_DIR.glob("fhvhv_tripdata_2024-*.parquet")),
}

file_summary = pd.DataFrame(
    {
        "service_type": service_type,
        "file_count": len(paths),
        "first_file": paths[0].name if paths else None,
        "last_file": paths[-1].name if paths else None,
    }
    for service_type, paths in raw_paths.items()
)

file_summary

,service_type,file_count,first_file,last_file
0,yellow,6,yellow_tripdata_2024-01.parquet,yellow_tripdata_2024-06.parquet
1,green,6,green_tripdata_2024-01.parquet,green_tripdata_2024-06.parquet
2,hvfhv,6,fhvhv_tripdata_2024-01.parquet,fhvhv_tripdata_2024-06.parquet


## Raw Row Counts

Count rows from parquet metadata rather than scanning the full datasets.
This gives the original data size for shape tracking.

In [44]:
raw_row_counts = pd.DataFrame(
    {
        "service_type": service_type,
        "raw_rows": sum(pq.ParquetFile(path).metadata.num_rows for path in paths),
    }
    for service_type, paths in raw_paths.items()
)

raw_row_counts

,service_type,raw_rows
0,yellow,20332093
1,green,339807
2,hvfhv,120864668


## Standardise Schemas

Yellow, green, and HVFHV files use different field names. These helper
functions rename the key columns into one common schema.

In [45]:
def standardise_service(raw_df, service_type):
    if service_type == "yellow":
        return raw_df.select(
            F.lit("yellow").alias("service_type"),
            F.col("tpep_pickup_datetime").alias("pickup_datetime"),
            F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
            F.col("PULocationID").alias("pickup_location_id"),
            F.col("DOLocationID").alias("dropoff_location_id"),
            F.col("trip_distance").alias("trip_miles"),
            F.col("total_amount").alias("earning_amount"),
        )

    if service_type == "green":
        return raw_df.select(
            F.lit("green").alias("service_type"),
            F.col("lpep_pickup_datetime").alias("pickup_datetime"),
            F.col("lpep_dropoff_datetime").alias("dropoff_datetime"),
            F.col("PULocationID").alias("pickup_location_id"),
            F.col("DOLocationID").alias("dropoff_location_id"),
            F.col("trip_distance").alias("trip_miles"),
            F.col("total_amount").alias("earning_amount"),
        )

    if service_type == "hvfhv":
        return raw_df.select(
            F.lit("hvfhv").alias("service_type"),
            F.col("pickup_datetime"),
            F.col("dropoff_datetime"),
            F.col("PULocationID").alias("pickup_location_id"),
            F.col("DOLocationID").alias("dropoff_location_id"),
            F.col("trip_miles"),
            F.col("driver_pay").alias("earning_amount"),
        )

    raise ValueError(f"Unknown service type: {service_type}")

## Feature Engineering and Cleaning

Add common time and earning-efficiency variables, then remove invalid or
extreme records using conservative rules.

In [46]:
def add_common_features(df):
    return (
        df
        .withColumn(
            "trip_duration_minutes",
            F.expr("timestampdiff(SECOND, pickup_datetime, dropoff_datetime)") / 60,
        )
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
        .withColumn("pickup_hour", F.date_trunc("hour", "pickup_datetime"))
        .withColumn("hour_of_day", F.hour("pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("pickup_datetime"))
        .withColumn(
            "earning_per_hour",
            F.col("earning_amount") / (F.col("trip_duration_minutes") / 60),
        )
    )


def clean_service_trips(df):
    return (
        df
        .filter(F.col("pickup_datetime") >= F.lit("2024-01-01"))
        .filter(F.col("pickup_datetime") < F.lit("2024-07-01"))
        .filter(F.col("pickup_location_id").between(1, 263))
        .filter(F.col("dropoff_location_id").between(1, 263))
        .filter(F.col("trip_duration_minutes") >= 2)
        .filter(F.col("trip_duration_minutes") <= 180)
        .filter(F.col("trip_miles") > 0)
        .filter(F.col("trip_miles") <= 100)
        .filter(F.col("earning_amount") > 0)
        .filter(F.col("earning_amount") <= 500)
    )

## Zone-Hour Aggregation

Aggregate each monthly file before combining services. This keeps the
large raw trip tables out of memory-heavy downstream steps.

In [47]:
def make_zone_hour(df):
    return (
        df
        .groupBy("service_type", "pickup_location_id", "pickup_hour")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("earning_amount").alias("avg_earning_amount"),
            F.avg("earning_per_hour").alias("avg_earning_per_hour"),
            F.avg("trip_miles").alias("avg_trip_miles"),
            F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        )
    )


def build_monthly_zone_hour(service_type, path):
    raw_df = spark.read.parquet(str(path))
    standardised = standardise_service(raw_df, service_type)
    featured = add_common_features(standardised)
    cleaned = clean_service_trips(featured)
    return make_zone_hour(cleaned)


def build_service_zone_hour(service_type):
    monthly_outputs = []

    for path in raw_paths[service_type]:
        month = path.stem.split("_")[-1]
        output_path = INTERIM_DIR / f"{service_type}_zone_hour_{month}"

        print(f"Processing {service_type} {month}")
        monthly_zone_hour = build_monthly_zone_hour(service_type, path)
        monthly_zone_hour.write.mode("overwrite").parquet(str(output_path))
        monthly_outputs.append(spark.read.parquet(str(output_path)))

    return reduce(lambda left, right: left.unionByName(right), monthly_outputs)

## Process TLC Trips

Build zone-hour tables for the three service types and save the combined
taxi/HVFHV table for reuse.

In [48]:
yellow_zone_hour = build_service_zone_hour("yellow")
green_zone_hour = build_service_zone_hour("green")
hvfhv_zone_hour = build_service_zone_hour("hvfhv")

zone_hour = (
    yellow_zone_hour
    .unionByName(green_zone_hour)
    .unionByName(hvfhv_zone_hour)
)

zone_hour_path = PROCESSED_DIR / "zone_hour_service_pickups"
zone_hour.write.mode("overwrite").parquet(str(zone_hour_path))
zone_hour = spark.read.parquet(str(zone_hour_path))

Processing yellow 2024-01


Processing yellow 2024-02
Processing yellow 2024-03
Processing yellow 2024-04


Processing yellow 2024-05
Processing yellow 2024-06
Processing green 2024-01
Processing green 2024-02
Processing green 2024-03
Processing green 2024-04
Processing green 2024-05
Processing green 2024-06
Processing hvfhv 2024-01


Processing hvfhv 2024-02


Processing hvfhv 2024-03


Processing hvfhv 2024-04


Processing hvfhv 2024-05


Processing hvfhv 2024-06


## Cleaning Summary

Summarise the number of retained trips after cleaning by adding up the
trip counts in the zone-hour table.

In [49]:
cleaned_row_counts = (
    zone_hour
    .groupBy("service_type")
    .agg(F.sum("trip_count").alias("cleaned_rows"))
    .orderBy("service_type")
)

cleaned_row_counts.show()

+------------+------------+
|service_type|cleaned_rows|
+------------+------------+
|       green|      312598|
|       hvfhv|   115816547|
|      yellow|    19347480|
+------------+------------+



## Subway Accessibility Features

Use MTA subway station points and TLC taxi zone polygons to measure
subway access for each taxi zone.

In [50]:
taxi_zones = gpd.read_file(TAXI_ZONES_PATH).to_crs("EPSG:2263")
mta_stations = pd.read_csv(MTA_PATH)

mta_gdf = gpd.GeoDataFrame(
    mta_stations,
    geometry=gpd.points_from_xy(
        mta_stations["GTFS Longitude"],
        mta_stations["GTFS Latitude"],
    ),
    crs="EPSG:4326",
).to_crs(taxi_zones.crs)

## Station Counts by Zone

Spatially join subway station points to taxi zone polygons, then count how
many stations fall inside each zone.

In [51]:
stations_in_zones = gpd.sjoin(
    mta_gdf,
    taxi_zones[["LocationID", "borough", "zone", "geometry"]],
    how="left",
    predicate="within",
)

station_counts = (
    stations_in_zones
    .groupby("LocationID")
    .agg(subway_station_count=("Station ID", "nunique"))
    .reset_index()
)

## Nearest Subway Distance

Some zones have no station inside their boundary but are still close to a
station. The nearest-station distance captures that extra accessibility
information.

In [52]:
zone_centroids = taxi_zones[["LocationID", "borough", "zone", "geometry"]].copy()
zone_centroids["geometry"] = zone_centroids.geometry.centroid

nearest_station = gpd.sjoin_nearest(
    zone_centroids,
    mta_gdf[["GTFS Stop ID", "Stop Name", "geometry"]],
    how="left",
    distance_col="nearest_subway_distance_ft",
)

nearest_station["nearest_subway_distance_km"] = (
    nearest_station["nearest_subway_distance_ft"] * 0.3048 / 1000
)

nearest_station = (
    nearest_station
    .sort_values(["LocationID", "nearest_subway_distance_km"])
    .drop_duplicates("LocationID")
    [["LocationID", "nearest_subway_distance_km", "Stop Name"]]
    .rename(columns={"Stop Name": "nearest_subway_station"})
)

## Accessibility Table

Combine the subway count and nearest-distance features, then classify
each taxi zone into high, medium, or low subway accessibility.

In [53]:
subway_access = (
    taxi_zones[["LocationID", "borough", "zone"]]
    .merge(station_counts, on="LocationID", how="left")
    .merge(nearest_station, on="LocationID", how="left")
)

subway_access["subway_station_count"] = (
    subway_access["subway_station_count"].fillna(0).astype(int)
)
subway_access["has_subway"] = subway_access["subway_station_count"] > 0

subway_access["subway_access_level"] = np.select(
    [
        subway_access["subway_station_count"] >= 2,
        subway_access["subway_station_count"] == 1,
        subway_access["nearest_subway_distance_km"] <= 0.5,
    ],
    ["high", "medium", "medium"],
    default="low",
)

subway_access_summary = (
    subway_access["subway_access_level"]
    .value_counts()
    .rename_axis("subway_access_level")
    .reset_index(name="zone_count")
)

subway_access_summary

,subway_access_level,zone_count
0,high,124
1,low,88
2,medium,51


## Join Taxi and Subway Features

Save the subway accessibility table, load it in Spark, and join it to the
zone-hour taxi/HVFHV aggregates.

In [54]:
subway_access_path = PROCESSED_DIR / "subway_access_by_taxi_zone.csv"
subway_access.to_csv(subway_access_path, index=False)

subway_access_spark = spark.read.csv(
    str(subway_access_path),
    header=True,
    inferSchema=True,
)

analysis_df = (
    zone_hour
    .join(
        subway_access_spark,
        zone_hour.pickup_location_id == subway_access_spark.LocationID,
        how="left",
    )
    .drop("LocationID")
)

missing_zone_rows = analysis_df.filter(F.col("zone").isNull()).count()
print(f"Rows without matched taxi zone: {missing_zone_rows}")

Rows without matched taxi zone: 0


## Save Analysis Dataset

Write the final processed table. Later notebooks should start from this
dataset instead of rereading all raw TLC files.

In [55]:
analysis_output_path = PROCESSED_DIR / "analysis_zone_hour"

analysis_df.write.mode("overwrite").parquet(str(analysis_output_path))

analysis_check = spark.read.parquet(str(analysis_output_path))
print(f"Saved analysis table to: {analysis_output_path}")
print(f"Rows in final analysis table: {analysis_check.count():,}")

Saved analysis table to: /mnt/d/ADS_proj1/data/processed/analysis_zone_hour
Rows in final analysis table: 1,663,511
